In [ ]:
"""
General approach:
For each row, my approach is to identify the cell type based on the unique markers that are present and 
identify the leftover unknowns with machine learning. For example, any rows that contain high endomucin
and low TCR, CD117, and CD11b based on + and - at the end of the marker is labeled as capillaries. This 
process is repeated for every cell-type identification using their respective unique marker. Then, to
identify rows that are still labeled as unknown, use random forest classifier and train on a dataset with 
known labels.
Optionally, KMeans could be applied to visualize patterns and validate assignments.

Key findings:
Labeling cell type strictly based on marker presence was only able to label half the dataset. This shows that
marker-based labeling alone is insufficient. Using random forest classifier helped to label cell type with > 0.7
precision. The recall was low for CFU-E (recall = 0.6), meaning that the model struggled to identify true CFU-E 
cells. The model had the most trouble correctly identifying erythrocytes. Despite this, for all other cell types,
The F1 score was mostly high, meaning that the cell type was able to be consistently recognized. The variance in F1
scores is likely, in part, due to class imbalance as rarer populations show lower performance.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

expression_data = pd.read_csv("/Users/janicesubroto/Desktop/Lu Lab/Python Quiz/Q3_gene_expression_data.csv")

,Unnamed: 0,Ter119,CD45,CD71,CD11b,TCR,CD117,CD31,CD19,CD105,...,Ter119: Membrane: Max,Ter119: Membrane: Mean,Ter119: Membrane: Median,Ter119: Membrane: Min,Ter119: Membrane: Std.Dev.,Ter119: Nucleus: Max,Ter119: Nucleus: Mean,Ter119: Nucleus: Median,Ter119: Nucleus: Min,Ter119: Nucleus: Std.Dev.
0,0,Ter119-,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,46,12.8974,11.0,1,9.7084,39,7.3611,5.0,0,8.5634
1,1,Ter119-,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,2,0.3816,0.0,0,0.5408,4,0.7103,1.0,0,0.6494
2,2,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,153,41.8919,28.0,5,36.9367,181,27.4603,16.0,2,34.0426
3,3,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,178,64.6522,52.0,11,44.1921,104,17.5859,9.0,0,23.3164
4,4,Ter119-,CD45+,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,37,8.5345,4.5,0,9.8697,28,7.6943,6.0,1,5.8766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,Ter119+,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,235,40.2075,16.0,0,57.6553,233,82.9023,76.0,10,43.0998
9996,9996,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,187,64.2000,46.0,5,55.5606,201,73.0901,53.0,7,57.5973
9997,9997,Ter119-,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,...,83,20.2245,15.0,0,17.5787,89,13.0250,4.0,0,21.5903
9998,9998,Ter119-,CD45+,CD71-,CD11b+,TCR-,CD117-,CD31-,CD19-,CD105-,...,106,32.6279,24.0,0,32.4639,112,28.5772,22.0,0,25.1370


In [ ]:

#if we want to categorize cells just based on qualitative data, we need to check off presence or absence of CD3+ TCR+ CD19- etc.
#Most markers are found on the membrane, so I would focus more on the membrane statistic
#Also might be best to focus on mean than other statistic

#need to characterize a threshold for the mean that is a decider for what is "high" or "low" expression
#I'm going to drop the stats columns that don't contain membrane and mean
expression_data = expression_data.drop(columns=['Unnamed: 0'])
exclude = ["Min", "Max", "Std.Dev.", "Median", "Nucleus", "Cytoplasm", "Cell"]
for e in exclude:
    cols_to_drop = expression_data.filter(regex=e, axis=1).columns
    expression_data = expression_data.drop(columns=cols_to_drop)


,Ter119,CD45,CD71,CD11b,TCR,CD117,CD31,CD19,CD105,Endomucin,...,F4_80: Membrane: Mean,Fcgr: Membrane: Mean,Foxp3: Membrane: Mean,IgD: Membrane: Mean,Lepr: Membrane: Mean,Ly6g: Membrane: Mean,MHCII: Membrane: Mean,Sca1: Membrane: Mean,TCR: Membrane: Mean,Ter119: Membrane: Mean
0,Ter119-,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,0.9487,6.0769,0.7436,7.7436,4.3333,11.7692,17.4103,8.4103,5.6154,12.8974
1,Ter119-,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,0.1184,9.4605,0.3026,1.8816,1.9211,10.7237,25.4474,19.4737,0.8158,0.3816
2,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,2.3784,11.5405,0.9730,4.2162,3.7027,5.7027,2.2162,2.2703,1.8108,41.8919
3,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,1.9565,10.6522,0.2609,3.3478,5.6739,2.4565,22.6957,4.8043,1.0652,64.6522
4,Ter119-,CD45+,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,0.9310,8.0172,0.8103,4.1379,25.5172,7.6207,18.8966,66.4310,1.4483,8.5345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Ter119+,CD45-,CD71-,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,0.8113,6.4151,0.7358,3.5472,3.6038,11.5849,1.9623,4.7170,1.1887,40.2075
9996,Ter119+,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,2.5333,6.6444,1.6444,4.2222,6.5111,15.5556,15.1778,3.8889,2.4444,64.2000
9997,Ter119-,CD45-,CD71+,CD11b-,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin-,...,7.6531,3.8367,5.1837,8.9592,2.0408,25.1429,6.2857,3.2857,4.5510,20.2245
9998,Ter119-,CD45+,CD71-,CD11b+,TCR-,CD117-,CD31-,CD19-,CD105-,Endomucin+,...,4.0698,22.3023,1.8837,11.7907,11.0233,37.1860,15.0000,3.7442,2.8372,32.6279


In [94]:
print(expression_data.columns)

Index(['Ter119', 'CD45', 'CD71', 'CD11b', 'TCR', 'CD117', 'CD31', 'CD19',
       'CD105', 'Endomucin', 'Lepr', 'B220', 'CD3', 'CD4', 'CD8', 'F4_80',
       'Ly6g', 'CD41', 'B220: Membrane: Mean', 'CD105: Membrane: Mean',
       'CD115: Membrane: Mean', 'CD117: Membrane: Mean',
       'CD11b: Membrane: Mean', 'CD11c: Membrane: Mean',
       'CD127: Membrane: Mean', 'CD19: Membrane: Mean',
       'CD21_35: Membrane: Mean', 'CD31: Membrane: Mean',
       'CD34: Membrane: Mean', 'CD38: Membrane: Mean', 'CD3: Membrane: Mean',
       'CD41: Membrane: Mean', 'CD44: Membrane: Mean', 'CD45: Membrane: Mean',
       'CD48: Membrane: Mean', 'CD49f: Membrane: Mean', 'CD4: Membrane: Mean',
       'CD5: Membrane: Mean', 'CD71: Membrane: Mean', 'CD8: Membrane: Mean',
       'CD90: Membrane: Mean', 'Endomucin: Membrane: Mean',
       'F4_80: Membrane: Mean', 'Fcgr: Membrane: Mean',
       'Foxp3: Membrane: Mean', 'IgD: Membrane: Mean', 'Lepr: Membrane: Mean',
       'Ly6g: Membrane: Mean', 'MHCII: Memb

In [96]:
to_check = {
"Arteries": ["CD31+", "TCR-", "CD117-", "CD11b-"],
"B Cells": ["CD19+", "B220+"], 
"Capillaries": ["Endomucin+", "TCR-", "CD117-", "CD11b-"],
"CD8 T Cells": ["CD3+", "CD8+", "TCR+", "CD4-"], 
"CFU-E": ["CD117+", "CD71+", "Ter119-", "CD41-"], 
"Endothelial Niche": ["CD105+", "CD45-", "Ter119-"], 
"Erythroblasts": ["Ter119+", "CD45-", "CD71+"], 
"Erythrocytes": ["Ter119+", "CD45-", "CD71-"], 
"Neutrophils": ["CD11b+", "Ly6g+", "F4_80-"], 
"Perivascular Niche": ["Lepr+", "CD45-", "Ter119-", "CD31-"] 
}


In [97]:
#for each row, check for candidates
#then check corresponding expression levels
#once verified, mark row with appropriate cell type
expression_data["Cell Type"] = "Unknown"

#iterate through rows
for i in range(len(expression_data)):
    for celltype, markers in to_check.items():
        match = True
        c_a = to_check[celltype] #extract array to check
        for marker in c_a:
            #take the name
            n = marker[:-1]
            cell_value = expression_data.loc[i,n]
            if cell_value != marker:
                match = False
                break
        
        if match:
            expression_data.loc[i, "Cell Type"] = celltype
            break



In [ ]:
print(expression_data["Cell Type"].value_counts())
#half are unknown
#use ML to identify unknowns

Cell Type
Unknown               5000
Erythroblasts         1400
Erythrocytes           800
Endothelial Niche      500
Neutrophils            500
Arteries               500
Perivascular Niche     400
Capillaries            400
B Cells                323
CFU-E                  125
CD8 T Cells             52
Name: count, dtype: int64

In [ ]:
labeled = expression_data[expression_data["Cell Type"] != "Unknown"]
unlabeled = expression_data[expression_data["Cell Type"] == "Unknown"]

X = labeled.select_dtypes(include=["number"])
y = labeled["Cell Type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) #1000 will be used for evaluation

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [100]:
#check performance
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

          Arteries       0.96      0.88      0.92       100
           B Cells       0.97      0.98      0.98        65
       CD8 T Cells       1.00      0.70      0.82        10
             CFU-E       0.94      0.60      0.73        25
       Capillaries       0.83      0.81      0.82        80
 Endothelial Niche       0.73      0.88      0.80       100
     Erythroblasts       0.80      0.90      0.85       280
      Erythrocytes       0.74      0.59      0.66       160
       Neutrophils       0.95      0.88      0.91       100
Perivascular Niche       0.94      0.94      0.94        80

          accuracy                           0.84      1000
         macro avg       0.88      0.82      0.84      1000
      weighted avg       0.84      0.84      0.84      1000

